# Indigo Flight Booking Assistant with LangGraph

A conversational AI assistant for booking Indigo airline flights using LangGraph and SQLite database integration.

## Section 1: Import Required Libraries and Set Up LangGraph

In [7]:
pip install -r requirements.txt

  Using cached langgraph-1.0.7-py3-none-any.whl.metadata (7.4 kB)
  Using cached openai-2.16.0-py3-none-any.whl.metadata (29 kB)
  Using cached streamlit-1.53.1-py3-none-any.whl.metadata (10 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.12.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached 

In [10]:
import sqlite3
import json
import os
from datetime import datetime
from typing import TypedDict, Optional, List, Dict, Any
from enum import Enum
import re

# LangGraph imports
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent

# LangChain imports
#from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Section 2: Load and Explore the Indigo Airline Database

In [13]:
# Database setup
DB_PATH = "/Users/vijendra/agentic-ai-usecases/advanced/flight-booking-assistant/indigo_airline.db"

def get_db_connection():
    """Create and return a database connection"""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

# Verify database exists
if os.path.exists(DB_PATH):
    print(f"✅ Database found at: {DB_PATH}")
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"✅ Database contains {len(tables)} tables:")
    for table in tables:
        print(f"  - {table[0]}")
    conn.close()
else:
    print(f"❌ Database not found at: {DB_PATH}")

# Airport code mapping
AIRPORT_CODES = {
    'jaipur': 'JAI',
    'mumbai': 'BOM',
    'delhi': 'DEL',
    'bangalore': 'BLR',
    'hyderabad': 'HYD',
    'kolkata': 'CCU',
    'kochi': 'COK',
    'pune': 'PNQ',
}

AIRPORT_NAMES = {
    'JAI': 'Jaipur International Airport',
    'BOM': 'Chhatrapati Shivaji International Airport',
    'DEL': 'Indira Gandhi International Airport',
    'BLR': 'Kempegowda International Airport',
    'HYD': 'Rajiv Gandhi International Airport',
    'CCU': 'Netaji Subhas Chandra Bose International Airport',
    'COK': 'Cochin International Airport',
    'PNQ': 'Pune Airport',
}

✅ Database found at: /Users/vijendra/agentic-ai-usecases/advanced/flight-booking-assistant/indigo_airline.db
✅ Database contains 17 tables:
  - Customers
  - FlightSchedule
  - DaysOfOperation
  - sqlite_sequence
  - PNRs
  - Bookings
  - Passengers
  - Itineraries
  - ItineraryLegs
  - PassengerBaggage
  - SpecialBaggage
  - FlightInstances
  - FlightDelays
  - Payments
  - Refunds
  - ConnectionRules
  - AuditLog


## Section 3: Define State and Tools for Flight Booking

In [14]:
# Define conversation states
class BookingState(TypedDict):
    """State for the flight booking conversation"""
    messages: List[BaseMessage]
    current_step: str
    service_selected: Optional[str]
    origin: Optional[str]
    destination: Optional[str]
    travel_date: Optional[str]
    return_date: Optional[str]
    journey_type: str  # 'oneway' or 'roundtrip'
    adults: int
    children: int
    children_ages: List[int]
    selected_flight: Optional[str]
    passenger_names: List[str]
    email: Optional[str]
    phone: Optional[str]
    pnr: Optional[str]
    booking_confirmed: bool
    available_flights: List[Dict[str, Any]]

# Helper functions for database queries
def get_flights(origin_code: str, destination_code: str) -> List[Dict[str, Any]]:
    """Get all flights between two airports"""
    conn = get_db_connection()
    cursor = conn.cursor()
    
    query = """
    SELECT flight_id, origin_airport_code, destination_airport_code,
           departure_time, arrival_time, flight_duration_minutes, 
           aircraft_type, seat_capacity, status
    FROM FlightSchedule
    WHERE origin_airport_code = ? AND destination_airport_code = ?
    ORDER BY departure_time
    """
    
    cursor.execute(query, (origin_code, destination_code))
    rows = cursor.fetchall()
    conn.close()
    
    flights = []
    for row in rows:
        flights.append({
            'flight_id': row['flight_id'],
            'origin': row['origin_airport_code'],
            'destination': row['destination_airport_code'],
            'departure_time': row['departure_time'],
            'arrival_time': row['arrival_time'],
            'duration_minutes': row['flight_duration_minutes'],
            'aircraft_type': row['aircraft_type'],
            'seats': row['seat_capacity'],
            'status': row['status']
        })
    
    return flights

def format_flight_list(flights: List[Dict[str, Any]], base_price: int = 5000) -> str:
    """Format flights for display"""
    if not flights:
        return "No flights available for this route."
    
    output = f"I found {len(flights)} onward flights for you on 10 February 2026\n"
    output += "=" * 50 + "\n"
    
    for idx, flight in enumerate(flights[:12], 1):  # Show first 12
        hours = flight['duration_minutes'] // 60
        mins = flight['duration_minutes'] % 60
        price = base_price + (idx * 500)
        
        output += f"Flight {idx}\n"
        output += f"{flight['departure_time']} -- {hours}h{mins}min -- {flight['arrival_time']}\n"
        output += f"Starts at rs{price}\n"
        output += "Non-stop\n"
        output += f"{flight['flight_id']}\n"
        output += "-" * 50 + "\n"
    
    output += "Choose the onward flight you wish to book.\n"
    output += "eg. flight 1, cheapest flight, 9:00AM\n"
    
    return output

print("✅ State and database query functions defined")

✅ State and database query functions defined


## Section 4: Create Conversation Flow with Multi-Step Routing

In [15]:
def greet_user(state: BookingState) -> BookingState:
    """Initial greeting"""
    greeting_msg = """Hello! I'm 6ESkai, your friendly AI assistant from Indigo.
How can I help you with our services today?
- Book a flight ticket
- Flight Status
- Web Check in"""
    
    state['messages'].append(AIMessage(content=greeting_msg))
    state['current_step'] = 'service_selection'
    return state

def process_service_selection(state: BookingState, user_input: str) -> BookingState:
    """Process service selection"""
    user_input_lower = user_input.lower()
    
    if 'book' in user_input_lower or 'ticket' in user_input_lower:
        state['service_selected'] = 'book_flight'
        next_msg = "Please let us know your destination"
        state['current_step'] = 'destination_input'
    elif 'status' in user_input_lower:
        state['service_selected'] = 'flight_status'
        next_msg = "Please provide your flight number or PNR"
        state['current_step'] = 'flight_status'
    elif 'check' in user_input_lower:
        state['service_selected'] = 'web_checkin'
        next_msg = "Please provide your PNR or booking reference"
        state['current_step'] = 'web_checkin'
    else:
        next_msg = "I didn't understand. Please select one of:\n- Book a flight ticket\n- Flight Status\n- Web Check in"
        state['current_step'] = 'service_selection'
    
    state['messages'].append(AIMessage(content=next_msg))
    return state

def get_destination(state: BookingState, user_input: str) -> BookingState:
    """Get destination from user"""
    destination = user_input.strip().lower()
    
    if destination in AIRPORT_CODES:
        state['destination'] = AIRPORT_CODES[destination]
        next_msg = "Please let us know your starting city"
        state['current_step'] = 'origin_input'
    else:
        next_msg = f"I don't recognize '{user_input}'. Please try: Mumbai, Delhi, Bangalore, Hyderabad, Kolkata, Kochi, or Pune"
        state['current_step'] = 'destination_input'
    
    state['messages'].append(AIMessage(content=next_msg))
    return state

def get_origin(state: BookingState, user_input: str) -> BookingState:
    """Get origin from user"""
    origin = user_input.strip().lower()
    
    if origin in AIRPORT_CODES:
        state['origin'] = AIRPORT_CODES[origin]
        next_msg = "Which day is your journey starting on? (eg. 10th Feb, Feb 10)"
        state['current_step'] = 'date_input'
    else:
        next_msg = f"I don't recognize '{user_input}'. Please try: Mumbai, Delhi, Bangalore, Hyderabad, Kolkata, Kochi, or Pune"
        state['current_step'] = 'origin_input'
    
    state['messages'].append(AIMessage(content=next_msg))
    return state

def get_travel_date(state: BookingState, user_input: str) -> BookingState:
    """Get travel date from user"""
    state['travel_date'] = user_input.strip()
    
    next_msg = "On which date will you conclude your travels?\n- [option] One-way only"
    state['current_step'] = 'return_date_input'
    
    state['messages'].append(AIMessage(content=next_msg))
    return state

def get_return_date(state: BookingState, user_input: str) -> BookingState:
    """Get return date or confirm one-way"""
    user_input_lower = user_input.lower().strip()
    
    if 'one' in user_input_lower or 'oneway' in user_input_lower:
        state['journey_type'] = 'oneway'
        state['return_date'] = None
    else:
        state['return_date'] = user_input.strip()
        state['journey_type'] = 'roundtrip'
    
    next_msg = """Can you please tell me the number of passengers?
eg. 2 adults, 1 child
Child age range:
EU region - between 2 and 16 years
Others - between 2 and 12 years"""
    
    state['current_step'] = 'passenger_count_input'
    state['messages'].append(AIMessage(content=next_msg))
    return state

def get_passenger_count(state: BookingState, user_input: str) -> BookingState:
    """Parse passenger count"""
    # Simple parsing - look for numbers
    text = user_input.lower()
    
    # Try to extract adults and children
    if 'adult' in text:
        adult_part = re.search(r'(\d+)\s*adult', text)
        if adult_part:
            state['adults'] = int(adult_part.group(1))
    
    if 'child' in text:
        child_part = re.search(r'(\d+)\s*child', text)
        if child_part:
            state['children'] = int(child_part.group(1))
    
    # Default: if only one number and no explicit adult/child, assume adult
    if state['adults'] == 0 and state['children'] == 0:
        numbers = re.findall(r'\d+', text)
        if numbers:
            state['adults'] = int(numbers[0])
    
    # If there are children, ask for their ages
    if state['children'] > 0:
        next_msg = f"""Please provide the ages of {state['children']} child passenger(s)"""
        state['current_step'] = 'child_age_input'
    else:
        next_msg = """Just to confirm how many child passengers are there?
Child age range:
EU region - between 2 and 16 years
Others - between 2 and 12 years
[option] No child passengers"""
        state['current_step'] = 'confirm_children'
    
    state['messages'].append(AIMessage(content=next_msg))
    return state

def confirm_children(state: BookingState, user_input: str) -> BookingState:
    """Confirm children count"""
    user_input_lower = user_input.lower().strip()
    
    if 'no' in user_input_lower or 'none' in user_input_lower:
        state['children'] = 0
    
    # Show booking summary
    next_msg = f"""Congratulations you are receiving a discounted fare by Indigo with our 6Exclusive offer..
Please review your travel details

Departure - {AIRPORT_NAMES.get(state['origin'], state['origin'])} ({state['origin']})
Destination: {AIRPORT_NAMES.get(state['destination'], state['destination'])} ({state['destination']})
Travel Date: {state['travel_date']}
Passenger Count - {state['adults']} Adult{'s' if state['adults'] > 1 else ''}{',' + str(state['children']) + ' Child' if state['children'] > 0 else ''}

to make changes, try-
eg. From Pune, return on Jan 26, 3 adults, no kids

Please review and confirm the booking specifics (yes/no)
[option] Yes
[option] No"""
    
    state['messages'].append(AIMessage(content=next_msg))
    state['current_step'] = 'booking_review'
    return state

print("✅ Conversation flow nodes defined")

✅ Conversation flow nodes defined


## Section 5: Implement Flight Search and Filtering

In [16]:
def confirm_booking_details(state: BookingState, user_input: str) -> BookingState:
    """Confirm booking details and fetch flights"""
    user_input_lower = user_input.lower().strip()
    
    if 'yes' in user_input_lower or 'confirm' in user_input_lower:
        # Fetch flights from database
        state['available_flights'] = get_flights(state['origin'], state['destination'])
        
        if state['available_flights']:
            flight_display = format_flight_list(state['available_flights'])
            state['messages'].append(AIMessage(content=flight_display))
            state['current_step'] = 'flight_selection'
        else:
            next_msg = "Sorry, no flights available for this route on that date. Please try different dates."
            state['messages'].append(AIMessage(content=next_msg))
            state['current_step'] = 'destination_input'
    else:
        next_msg = "Please let us know your destination"
        state['messages'].append(AIMessage(content=next_msg))
        state['origin'] = None
        state['destination'] = None
        state['current_step'] = 'destination_input'
    
    return state

def select_flight(state: BookingState, user_input: str) -> BookingState:
    """Process flight selection"""
    user_input = user_input.strip().lower()
    
    flight_num = None
    
    # Try to parse flight number
    if 'flight' in user_input:
        num_match = re.search(r'flight\s*(\d+)', user_input)
        if num_match:
            flight_num = int(num_match.group(1)) - 1
    elif user_input.isdigit():
        flight_num = int(user_input) - 1
    
    if flight_num is not None and 0 <= flight_num < len(state['available_flights']):
        selected = state['available_flights'][flight_num]
        state['selected_flight'] = selected['flight_id']
        
        hours = selected['duration_minutes'] // 60
        mins = selected['duration_minutes'] % 60
        
        next_msg = f"""please review onward flight details
Departure: ({AIRPORT_NAMES.get(state['origin'], state['origin'])}) {state['origin']}
Destination: {AIRPORT_NAMES.get(state['destination'], state['destination'])} ({state['destination']})
Travel Date: 10th Feb 2026
{selected['departure_time']} -- {hours}h{mins}min -- {selected['arrival_time']}
Starts at rs6031
Non-stop
{selected['flight_id']}

Please review and confirm your Indigo {selected['flight_id']} flight
[option] Yes
[option] No"""
        
        state['messages'].append(AIMessage(content=next_msg))
        state['current_step'] = 'flight_confirmation'
    else:
        next_msg = "Please select a valid flight number (eg. flight 1, flight 2)"
        state['messages'].append(AIMessage(content=next_msg))
    
    return state

print("✅ Flight search and selection functions defined")

✅ Flight search and selection functions defined


## Section 6: Handle Passenger Details Collection

In [17]:
def confirm_flight_selection(state: BookingState, user_input: str) -> BookingState:
    """Confirm selected flight and proceed to consent"""
    user_input_lower = user_input.lower().strip()
    
    if 'yes' in user_input_lower or 'confirm' in user_input_lower:
        next_msg = """To keep you informed about our products and services, we would like your consent to communicate with you on Whatsapp.
By confirming Yes, you agree to IndiGo's Privacy Policy and Consent Management Policy
https://www.goIndigo.in/Information/privacy.html
[option] Yes
[option] No"""
        state['messages'].append(AIMessage(content=next_msg))
        state['current_step'] = 'whatsapp_consent'
    else:
        next_msg = "Flight selection cancelled. Please select another flight or start over."
        state['messages'].append(AIMessage(content=next_msg))
        state['current_step'] = 'flight_selection'
    
    return state

def get_whatsapp_consent(state: BookingState, user_input: str) -> BookingState:
    """Get WhatsApp consent and proceed to passenger details"""
    user_input_lower = user_input.lower().strip()
    
    # Store consent (Yes/No)
    if 'yes' in user_input_lower:
        consent = True
    else:
        consent = False
    
    next_msg = """Okay, continuing with the process,
Can you please tell me the full name of all the passengers
eg: Mr./Mrs/Miss First Name Last Name
Please provide the names together in one go"""
    
    state['messages'].append(AIMessage(content=next_msg))
    state['current_step'] = 'passenger_names'
    return state

def get_passenger_names(state: BookingState, user_input: str) -> BookingState:
    """Get passenger names"""
    # Parse names - simple approach: split by comma or 'and'
    names = [n.strip() for n in re.split(r'[,\s+and\s+]', user_input) if n.strip()]
    state['passenger_names'] = names
    
    next_msg = "Please provide your email address in the correct format.\neg: email_id@website.com"
    state['messages'].append(AIMessage(content=next_msg))
    state['current_step'] = 'email_input'
    return state

def get_email(state: BookingState, user_input: str) -> BookingState:
    """Get and validate email"""
    email = user_input.strip()
    
    # Simple email validation
    if '@' in email and '.' in email:
        state['email'] = email
        next_msg = "Please provide your contact phone number (10 digits)"
        state['current_step'] = 'phone_input'
    else:
        next_msg = "Invalid email format. Please try again: email_id@website.com"
        state['current_step'] = 'email_input'
    
    state['messages'].append(AIMessage(content=next_msg))
    return state

def get_phone(state: BookingState, user_input: str) -> BookingState:
    """Get and validate phone number"""
    phone = re.sub(r'\D', '', user_input)  # Remove non-digits
    
    if len(phone) >= 10:
        state['phone'] = phone
        state['current_step'] = 'payment_review'
    else:
        next_msg = "Invalid phone number. Please provide a 10-digit number."
        state['messages'].append(AIMessage(content=next_msg))
        state['current_step'] = 'phone_input'
        return state
    
    # Don't add message yet, proceed to payment review
    return state

print("✅ Passenger details collection functions defined")

✅ Passenger details collection functions defined


## Section 7: Build Booking Confirmation Logic

In [18]:
import random
import string

def generate_pnr():
    """Generate a 6-character PNR code"""
    return ''.join(random.choices(string.ascii_uppercase + string.digits, k=6))

def show_payment_review(state: BookingState) -> BookingState:
    """Show payment review and booking summary"""
    state['pnr'] = generate_pnr()
    
    # Get selected flight details
    selected_flight = None
    for flight in state['available_flights']:
        if flight['flight_id'] == state['selected_flight']:
            selected_flight = flight
            break
    
    if not selected_flight:
        state['messages'].append(AIMessage(content="Error: Flight details not found"))
        return state
    
    # Calculate fare
    base_fare = 6031
    total_fare = base_fare * state['adults']
    if state['children'] > 0:
        total_fare += (base_fare * 0.5) * state['children']
    
    hours = selected_flight['duration_minutes'] // 60
    mins = selected_flight['duration_minutes'] % 60
    
    payment_msg = f"""Review Your Booking
-----------------------------------

{state['origin']} ↔️ {state['destination']} (ONWARD)

🗓️ 10-02-2026 

{' '.join(state['passenger_names']) if state['passenger_names'] else 'Passenger'}

✈️ Onward flight - 10-02-2026 
{state['selected_flight']}
{selected_flight['departure_time']} - {selected_flight['arrival_time']} ({hours}h {mins}min)
Nonstop 
{AIRPORT_NAMES.get(state['origin'], state['origin'])} ➡️ {AIRPORT_NAMES.get(state['destination'], state['destination'])}

Payment Details
-----------------------------------
Adult(s)         {state['adults']} x ₹{base_fare:,}    ₹{base_fare * state['adults']:,}
{'Child(ren)         ' + str(state['children']) + ' x ₹' + str(int(base_fare * 0.5)) + '    ₹' + str(int(base_fare * 0.5 * state['children'])) if state['children'] > 0 else ''}
-----------------------------------
Total                     ₹{int(total_fare):,}

* Convenience fee may apply
 
Please proceed with payment via WhatsApp mobile to confirm your booking.
Tap the Review and pay button to complete payment. Your PNR will be sent to 918600911152 after successful payment.

💡 Check for available bank offers or promo codes on the payment page to save more!

By clicking Continue, you confirm you've read and understood the fare restrictions and agree to IndiGo's Privacy Policy: https://www.goindigo.in/information/privacy.html
The payment link is valid for 10 minutes."""
    
    state['messages'].append(AIMessage(content=payment_msg))
    state['booking_confirmed'] = True
    state['current_step'] = 'completed'
    
    return state

def process_user_input(state: BookingState, user_input: str) -> BookingState:
    """Route user input to appropriate handler based on current step"""
    state['messages'].append(HumanMessage(content=user_input))
    
    current_step = state['current_step']
    
    if current_step == 'greeting':
        state = greet_user(state)
    elif current_step == 'service_selection':
        state = process_service_selection(state, user_input)
    elif current_step == 'destination_input':
        state = get_destination(state, user_input)
    elif current_step == 'origin_input':
        state = get_origin(state, user_input)
    elif current_step == 'date_input':
        state = get_travel_date(state, user_input)
    elif current_step == 'return_date_input':
        state = get_return_date(state, user_input)
    elif current_step == 'passenger_count_input':
        state = get_passenger_count(state, user_input)
    elif current_step == 'confirm_children':
        state = confirm_children(state, user_input)
    elif current_step == 'booking_review':
        state = confirm_booking_details(state, user_input)
    elif current_step == 'flight_selection':
        state = select_flight(state, user_input)
    elif current_step == 'flight_confirmation':
        state = confirm_flight_selection(state, user_input)
    elif current_step == 'whatsapp_consent':
        state = get_whatsapp_consent(state, user_input)
    elif current_step == 'passenger_names':
        state = get_passenger_names(state, user_input)
    elif current_step == 'email_input':
        state = get_email(state, user_input)
    elif current_step == 'phone_input':
        state = get_phone(state, user_input)
        if state['current_step'] == 'payment_review':
            state = show_payment_review(state)
    
    return state

print("✅ Booking confirmation and routing functions defined")

✅ Booking confirmation and routing functions defined


## Section 8: Test the Flight Booking Assistant

In [19]:
def create_initial_state() -> BookingState:
    """Create initial booking state"""
    return {
        'messages': [],
        'current_step': 'greeting',
        'service_selected': None,
        'origin': None,
        'destination': None,
        'travel_date': None,
        'return_date': None,
        'journey_type': 'oneway',
        'adults': 0,
        'children': 0,
        'children_ages': [],
        'selected_flight': None,
        'passenger_names': [],
        'email': None,
        'phone': None,
        'pnr': None,
        'booking_confirmed': False,
        'available_flights': []
    }

def run_flight_booking_assistant():
    """Run the flight booking assistant conversation"""
    state = create_initial_state()
    
    # Start with greeting
    state = greet_user(state)
    
    # Simulate conversation flow
    test_inputs = [
        "Book a flight ticket",
        "Mumbai",
        "Jaipur",
        "10th Feb",
        "One-way only",
        "1 adult",
        "No child passengers",
        "Yes",
        "Flight 2",
        "Yes",
        "Yes",
        "Mr. Vijendra Jain",
        "vijendra.1893@gmail.com",
        "9876543210"
    ]
    
    print("=" * 70)
    print("🛫 INDIGO FLIGHT BOOKING ASSISTANT DEMO 🛫")
    print("=" * 70)
    
    for i, user_input in enumerate(test_inputs, 1):
        print(f"\n👤 User ({i}): {user_input}")
        state = process_user_input(state, user_input)
        
        # Get the last message (assistant response)
        if state['messages']:
            last_msg = state['messages'][-1]
            if hasattr(last_msg, 'content'):
                print(f"🤖 Assistant: {last_msg.content}")
        
        if state['current_step'] == 'completed':
            print("\n✅ Booking process completed!")
            break
    
    print("\n" + "=" * 70)
    print("📊 BOOKING SUMMARY")
    print("=" * 70)
    print(f"Origin: {state['origin']}")
    print(f"Destination: {state['destination']}")
    print(f"Travel Date: {state['travel_date']}")
    print(f"Passengers: {state['adults']} adult(s), {state['children']} child(ren)")
    print(f"Selected Flight: {state['selected_flight']}")
    print(f"Passenger Names: {', '.join(state['passenger_names'])}")
    print(f"Email: {state['email']}")
    print(f"Phone: {state['phone']}")
    print(f"PNR Code: {state['pnr']}")
    print(f"Booking Confirmed: {'✅ Yes' if state['booking_confirmed'] else '❌ No'}")
    print("=" * 70)
    
    return state

# Run the assistant
final_state = run_flight_booking_assistant()

🛫 INDIGO FLIGHT BOOKING ASSISTANT DEMO 🛫

👤 User (1): Book a flight ticket
🤖 Assistant: Please let us know your destination

👤 User (2): Mumbai
🤖 Assistant: Please let us know your starting city

👤 User (3): Jaipur
🤖 Assistant: Which day is your journey starting on? (eg. 10th Feb, Feb 10)

👤 User (4): 10th Feb
🤖 Assistant: On which date will you conclude your travels?
- [option] One-way only

👤 User (5): One-way only
🤖 Assistant: Can you please tell me the number of passengers?
eg. 2 adults, 1 child
Child age range:
EU region - between 2 and 16 years
Others - between 2 and 12 years

👤 User (6): 1 adult
🤖 Assistant: Just to confirm how many child passengers are there?
Child age range:
EU region - between 2 and 16 years
Others - between 2 and 12 years
[option] No child passengers

👤 User (7): No child passengers
🤖 Assistant: Congratulations you are receiving a discounted fare by Indigo with our 6Exclusive offer..
Please review your travel details

Departure - Jaipur International Airport

## Interactive Chat with the Flight Booking Assistant

Use the function below to have a real conversation with the flight booking assistant.

In [20]:
# Global state for interactive chat
chat_state = None

def start_interactive_chat():
    """Start an interactive chat session"""
    global chat_state
    chat_state = create_initial_state()
    chat_state = greet_user(chat_state)
    print("\n🤖 " + chat_state['messages'][-1].content)
    print("\n" + "=" * 70)

def chat(user_message: str):
    """Send a message in the ongoing chat"""
    global chat_state
    
    if chat_state is None:
        start_interactive_chat()
    
    # Process user input
    chat_state = process_user_input(chat_state, user_message)
    
    # Get assistant response
    if chat_state['messages']:
        last_msg = chat_state['messages'][-1]
        print(f"\n👤 You: {user_message}")
        print(f"\n🤖 Assistant:\n{last_msg.content}")
        
        if chat_state['current_step'] == 'completed':
            print("\n✅ Booking completed! Thank you for choosing Indigo.")
    
    return chat_state

# Start the interactive session
print("=" * 70)
print("Welcome to Indigo Flight Booking Assistant!")
print("=" * 70)
print("Type chat('your message') to interact with the assistant.")
print("Example: chat('Book a flight ticket')")
print("=" * 70)

start_interactive_chat()

Welcome to Indigo Flight Booking Assistant!
Type chat('your message') to interact with the assistant.
Example: chat('Book a flight ticket')

🤖 Hello! I'm 6ESkai, your friendly AI assistant from Indigo.
How can I help you with our services today?
- Book a flight ticket
- Flight Status
- Web Check in

